<a href="https://colab.research.google.com/github/PrashanthBhaskara/KalshiCorrelationForecast/blob/main/AI_Brier_Score_24hr_advance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os, re, glob
import numpy as np
import pandas as pd
import requests

In [2]:
files = sorted(glob.glob("/content/kalshi-price-history-*.csv"))
assert files, "No files found. Upload kalshi-price-history-*.csv into /content."

def event_from_filename(path: str) -> str:
    # kalshi-price-history-kxllm1-25dec31-day.csv -> KXLLM1-25DEC31
    base = os.path.basename(path).lower()
    m = re.search(r"kalshi-price-history-(kxllm1-\d{2}[a-z]{3}\d{2})", base)
    if not m:
        raise ValueError(f"Could not parse event from filename: {base}")
    return m.group(1).upper()

dfs = []
for f in files:
    df = pd.read_csv(f)
    df["event_ticker"] = event_from_filename(f)
    dfs.append(df)

raw = pd.concat(dfs, ignore_index=True)

print("Loaded files:", len(files))
print("Rows:", len(raw))
print("Events:", sorted(raw["event_ticker"].unique()))
print("Columns:", raw.columns.tolist())
raw.head()

Loaded files: 7
Rows: 457
Events: ['KXLLM1-25DEC31', 'KXLLM1-26FEB07', 'KXLLM1-26FEB14', 'KXLLM1-26FEB21', 'KXLLM1-26JAN17', 'KXLLM1-26JAN24', 'KXLLM1-26JAN31']
Columns: ['timestamp', 'Gemini', 'Claude', 'Qwen', 'DeepSeek', 'LLaMA', 'ChatGPT', 'Grok', 'event_ticker', 'Ernie', 'Dola']


,timestamp,Gemini,Claude,Qwen,DeepSeek,LLaMA,ChatGPT,Grok,event_ticker,Ernie,Dola
0,2024-11-06T00:00:00Z,NaN,2.00,NaN,NaN,NaN,99.00,2.00,KXLLM1-25DEC31,NaN,NaN
1,2024-11-07T00:00:00Z,11.93,4.08,NaN,NaN,NaN,85.78,60.84,KXLLM1-25DEC31,NaN,NaN
2,2024-11-08T00:00:00Z,8.00,12.82,NaN,NaN,NaN,65.45,22.55,KXLLM1-25DEC31,NaN,NaN
3,2024-11-09T00:00:00Z,13.44,12.83,NaN,NaN,NaN,66.29,13.40,KXLLM1-25DEC31,NaN,NaN
4,2024-11-10T00:00:00Z,15.99,12.00,NaN,NaN,NaN,66.81,12.00,KXLLM1-25DEC31,NaN,NaN


In [4]:
# Try ISO/standard parse first
raw["timestamp_parsed"] = pd.to_datetime(raw["timestamp"], utc=True, errors="coerce")

# If that failed for many rows, assume unix seconds
if raw["timestamp_parsed"].isna().mean() > 0.5:
    raw["timestamp_parsed"] = pd.to_datetime(raw["timestamp"], unit="s", utc=True, errors="coerce")

assert raw["timestamp_parsed"].notna().all(), "Timestamp parsing failed for some rows."

TIME_COL = "timestamp_parsed"

# Infer model columns: numeric columns except metadata
non_model_cols = {"timestamp", "timestamp_parsed", "event_ticker"}
model_cols = [c for c in raw.columns if c not in non_model_cols]

# Convert to numeric and scale cents -> probs (0..1)
for c in model_cols:
    raw[c] = pd.to_numeric(raw[c], errors="coerce") / 100.0

print("Model columns:", model_cols)
raw[[TIME_COL, "event_ticker"] + model_cols[:5]].head()

Model columns: ['Gemini', 'Claude', 'Qwen', 'DeepSeek', 'LLaMA', 'ChatGPT', 'Grok', 'Ernie', 'Dola']


,timestamp_parsed,event_ticker,Gemini,Claude,Qwen,DeepSeek,LLaMA
0,2024-11-06 00:00:00+00:00,KXLLM1-25DEC31,NaN,0.0200,NaN,NaN,NaN
1,2024-11-07 00:00:00+00:00,KXLLM1-25DEC31,0.1193,0.0408,NaN,NaN,NaN
2,2024-11-08 00:00:00+00:00,KXLLM1-25DEC31,0.0800,0.1282,NaN,NaN,NaN
3,2024-11-09 00:00:00+00:00,KXLLM1-25DEC31,0.1344,0.1283,NaN,NaN,NaN
4,2024-11-10 00:00:00+00:00,KXLLM1-25DEC31,0.1599,0.1200,NaN,NaN,NaN


In [7]:
import time, random
import pandas as pd
import requests

BASE_URL = "https://api.elections.kalshi.com/trade-api/v2"
session = requests.Session()
session.headers.update({"accept": "application/json"})

# ---- rate limit / retry wrapper ----
def kalshi_get(path: str, params: dict, max_retries: int = 8, base_sleep: float = 0.6):
    """
    GET with exponential backoff + jitter for 429/5xx.
    """
    url = f"{BASE_URL}{path}"
    for attempt in range(max_retries):
        r = session.get(url, params=params, timeout=30)
        if r.status_code == 429 or (500 <= r.status_code < 600):
            # honor Retry-After if provided
            retry_after = r.headers.get("Retry-After")
            if retry_after:
                sleep_s = float(retry_after)
            else:
                sleep_s = base_sleep * (2 ** attempt)
            sleep_s = sleep_s + random.random() * 0.25  # jitter
            time.sleep(sleep_s)
            continue
        r.raise_for_status()
        return r.json()
    # last attempt
    r.raise_for_status()

def get_event_markets(event_ticker: str, limit: int = 1000):
    """
    One call (plus pagination if cursor exists), with backoff.
    """
    all_markets = []
    cursor = None
    while True:
        params = {"event_ticker": event_ticker, "limit": limit}
        if cursor:
            params["cursor"] = cursor
        payload = kalshi_get("/markets", params=params)
        all_markets.extend(payload.get("markets", []) or [])
        cursor = payload.get("cursor")
        if not cursor:
            break
        # small sleep between pages to be polite
        time.sleep(0.2)
    return all_markets

# ---- parse close time from markets ----
def extract_close_time(markets: list[dict]) -> pd.Timestamp | None:
    if not markets:
        return None
    # Try any market that has close_time
    for m in markets:
        v = m.get("close_time") or m.get("close_ts")
        if v is None or v == "":
            continue
        try:
            return pd.to_datetime(v, unit="s", utc=True)
        except Exception:
            t = pd.to_datetime(v, utc=True, errors="coerce")
            if pd.notna(t):
                return t
    return None

# ---- map winner market to model col ----
SUFFIX_ALIAS = {
    "GOOG": "Gemini",
    "ANTH": "Claude",
    "OPEN": "ChatGPT",
    "XAI":  "Grok",
    "QWEN": "Qwen",
    "META": "LLaMA",
    "ERN":  "Ernie",
    "DEEP": "DeepSeek",
    "DOLA": "Dola",
}

def market_to_model_name(market: dict, model_columns: list[str]) -> str | None:
    txt = " ".join([
        str(market.get("title","")),
        str(market.get("subtitle","")),
        str(market.get("ticker","")),
    ]).lower()

    for m in model_columns:
        if m.lower() in txt:
            return m

    t = str(market.get("ticker",""))
    if "-" in t:
        suf = t.split("-")[-1].upper()
        mapped = SUFFIX_ALIAS.get(suf)
        if mapped and mapped in model_columns:
            return mapped

    return None

def extract_winner_model(markets: list[dict], model_columns: list[str]) -> str | None:
    yes_settled = [m for m in markets if m.get("result") == "yes"]
    if not yes_settled:
        return None
    winner_market = yes_settled[0]
    return market_to_model_name(winner_market, model_columns)

# ---- cache so each event only hits API once ----
EVENT_CACHE = {}  # event_ticker -> {"markets":..., "close_time":..., "winner_model":...}

def get_event_info(event_ticker: str, model_columns: list[str]):
    if event_ticker in EVENT_CACHE:
        return EVENT_CACHE[event_ticker]

    mkts = get_event_markets(event_ticker)
    info = {
        "markets": mkts,
        "close_time": extract_close_time(mkts),
        "winner_model": extract_winner_model(mkts, model_columns),
    }
    EVENT_CACHE[event_ticker] = info

    # small sleep between events to reduce 429 risk
    time.sleep(0.35)
    return info

In [8]:
rows_24h = []
meta = []

for event in sorted(raw["event_ticker"].unique()):
    close_time = get_event_close_time(event)
    if close_time is None or pd.isna(close_time):
        print(f"Skipping {event} (no close_time)")
        continue

    cutoff = close_time - pd.Timedelta(hours=24)

    df_e = raw[raw["event_ticker"] == event].copy()
    df_e = df_e[df_e[TIME_COL] <= cutoff]
    if df_e.empty:
        print(f"Skipping {event} (no rows <= cutoff {cutoff})")
        continue

    snap = df_e.sort_values(TIME_COL).iloc[-1]
    rows_24h.append(snap)

    meta.append({"event_ticker": event, "close_time": close_time, "cutoff": cutoff, "snapshot_time": snap[TIME_COL]})

final_probs_24h = pd.DataFrame(rows_24h)
meta_24h = pd.DataFrame(meta)

print("Snapshots:", len(final_probs_24h))
display(meta_24h.sort_values("event_ticker"))

Snapshots: 7


,event_ticker,close_time,cutoff,snapshot_time
0,KXLLM1-25DEC31,2025-12-31 15:00:00+00:00,2025-12-30 15:00:00+00:00,2025-12-30 00:00:00+00:00
1,KXLLM1-26FEB07,2026-02-07 15:00:00+00:00,2026-02-06 15:00:00+00:00,2026-02-06 00:00:00+00:00
2,KXLLM1-26FEB14,2026-02-14 15:00:00+00:00,2026-02-13 15:00:00+00:00,2026-02-13 00:00:00+00:00
3,KXLLM1-26FEB21,2026-02-21 15:00:00+00:00,2026-02-20 15:00:00+00:00,2026-02-20 00:00:00+00:00
4,KXLLM1-26JAN17,2026-01-17 15:00:00+00:00,2026-01-16 15:00:00+00:00,2026-01-16 00:00:00+00:00
5,KXLLM1-26JAN24,2026-01-24 15:00:00+00:00,2026-01-23 15:00:00+00:00,2026-01-23 00:00:00+00:00
6,KXLLM1-26JAN31,2026-01-31 15:00:00+00:00,2026-01-30 15:00:00+00:00,2026-01-30 00:00:00+00:00


In [9]:
# Only keep model cols that actually exist / have any data
model_cols_24h = [m for m in model_cols if final_probs_24h[m].notna().any()]

winners = []
settled = []

for _, r in final_probs_24h.iterrows():
    event = r["event_ticker"]
    try:
        winners.append(get_winner_model_for_event(event, model_cols_24h))
        settled.append(True)
    except Exception as e:
        print(f"Skipping {event} (unsettled or mapping failed): {e}")
        winners.append(None)
        settled.append(False)

final_probs_24h["winner_model"] = winners
final_probs_24h["is_settled"] = settled

final_probs_24h_settled = final_probs_24h[final_probs_24h["is_settled"]].copy()

def multiclass_brier(row, cols):
    w = row["winner_model"]
    s = 0.0
    for m in cols:
        p = row[m]
        if pd.isna(p):
            continue
        y = 1.0 if m == w else 0.0
        s += (p - y) ** 2
    return s

final_probs_24h_settled["brier_multiclass_24h"] = final_probs_24h_settled.apply(
    lambda r: multiclass_brier(r, model_cols_24h), axis=1
)

# Per-model binary Brier @24h
rows = []
for m in model_cols_24h:
    y = (final_probs_24h_settled["winner_model"] == m).astype(float)
    p = final_probs_24h_settled[m].astype(float)
    rows.append({"model": m, "brier_binary_mean_24h": float(((p - y) ** 2).mean())})

per_model_brier_24h = pd.DataFrame(rows).sort_values("brier_binary_mean_24h")

print("Mean multiclass Brier @24h:", final_probs_24h_settled["brier_multiclass_24h"].mean())
display(final_probs_24h_settled[["event_ticker", TIME_COL, "winner_model", "brier_multiclass_24h"]].sort_values("event_ticker"))
display(per_model_brier_24h)

Mean multiclass Brier @24h: 0.28728518285714283


,event_ticker,timestamp_parsed,winner_model,brier_multiclass_24h
385,KXLLM1-25DEC31,2025-12-30 00:00:00+00:00,Gemini,0.001884
392,KXLLM1-26FEB07,2026-02-06 00:00:00+00:00,Claude,1.926604
399,KXLLM1-26FEB14,2026-02-13 00:00:00+00:00,Claude,0.020938
406,KXLLM1-26FEB21,2026-02-20 00:00:00+00:00,Claude,0.056275
415,KXLLM1-26JAN17,2026-01-16 00:00:00+00:00,Gemini,0.002394
422,KXLLM1-26JAN24,2026-01-23 00:00:00+00:00,Gemini,0.001688
455,KXLLM1-26JAN31,2026-01-30 00:00:00+00:00,Gemini,0.001212


,model,brier_binary_mean_24h
3,DeepSeek,0.000100
7,Ernie,0.000100
8,Dola,0.000100
4,LLaMA,0.000100
2,Qwen,0.000100
5,ChatGPT,0.000306
6,Grok,0.000420
1,Claude,0.141861
0,Gemini,0.144384


In [10]:
# Only keep model cols that actually exist / have any data
model_cols_24h = [m for m in model_cols if final_probs_24h[m].notna().any()]

winners = []
settled = []

for _, r in final_probs_24h.iterrows():
    event = r["event_ticker"]
    try:
        winners.append(get_winner_model_for_event(event, model_cols_24h))
        settled.append(True)
    except Exception as e:
        print(f"Skipping {event} (unsettled or mapping failed): {e}")
        winners.append(None)
        settled.append(False)

final_probs_24h["winner_model"] = winners
final_probs_24h["is_settled"] = settled

final_probs_24h_settled = final_probs_24h[final_probs_24h["is_settled"]].copy()

def multiclass_brier(row, cols):
    w = row["winner_model"]
    s = 0.0
    for m in cols:
        p = row[m]
        if pd.isna(p):
            continue
        y = 1.0 if m == w else 0.0
        s += (p - y) ** 2
    return s

final_probs_24h_settled["brier_multiclass_24h"] = final_probs_24h_settled.apply(
    lambda r: multiclass_brier(r, model_cols_24h), axis=1
)

# Per-model binary Brier @24h
rows = []
for m in model_cols_24h:
    y = (final_probs_24h_settled["winner_model"] == m).astype(float)
    p = final_probs_24h_settled[m].astype(float)
    rows.append({"model": m, "brier_binary_mean_24h": float(((p - y) ** 2).mean())})

per_model_brier_24h = pd.DataFrame(rows).sort_values("brier_binary_mean_24h")

print("Mean multiclass Brier @24h:", final_probs_24h_settled["brier_multiclass_24h"].mean())
display(final_probs_24h_settled[["event_ticker", TIME_COL, "winner_model", "brier_multiclass_24h"]].sort_values("event_ticker"))
display(per_model_brier_24h)

Mean multiclass Brier @24h: 0.28728518285714283


,event_ticker,timestamp_parsed,winner_model,brier_multiclass_24h
385,KXLLM1-25DEC31,2025-12-30 00:00:00+00:00,Gemini,0.001884
392,KXLLM1-26FEB07,2026-02-06 00:00:00+00:00,Claude,1.926604
399,KXLLM1-26FEB14,2026-02-13 00:00:00+00:00,Claude,0.020938
406,KXLLM1-26FEB21,2026-02-20 00:00:00+00:00,Claude,0.056275
415,KXLLM1-26JAN17,2026-01-16 00:00:00+00:00,Gemini,0.002394
422,KXLLM1-26JAN24,2026-01-23 00:00:00+00:00,Gemini,0.001688
455,KXLLM1-26JAN31,2026-01-30 00:00:00+00:00,Gemini,0.001212


,model,brier_binary_mean_24h
3,DeepSeek,0.000100
7,Ernie,0.000100
8,Dola,0.000100
4,LLaMA,0.000100
2,Qwen,0.000100
5,ChatGPT,0.000306
6,Grok,0.000420
1,Claude,0.141861
0,Gemini,0.144384
